# Lab 9: A/B Testing — Designing the Data-Generating Process
## ECON 3916: Applied Machine Learning for Economists
### Guided Construction Lab | 65 min Core + 20 min Write it yourself + 15 min Extension + take-home Challenge

---

**Learning Objectives:**
- Distinguish experimental from observational estimates and quantify selection bias
- Conduct a permutation test to assess whether a treatment effect is statistically significant
- Use bootstrap confidence intervals to quantify uncertainty around the Average Treatment Effect (ATE)
- Perform power analysis to determine the sample size required to detect a meaningful effect
- Write a sample-size function from an effect and a standard deviation, and use it to plan a rerun of an A/B test

**Datasets:** LaLonde National Supported Work (NSW) Demonstration + Cookie Cats A/B Test (Kaggle)

**Prerequisites:** the Part 0 sections of Labs 1-5, and the labs before this one.

**How this lab works:** Parts 1-3 are GUIDED: run each cell as it is, then answer the questions. Part 4 runs one guided step, then has five blanks to fill in across three cells. "Write it yourself" is required: a short function you write from a blank cell. The Extension is optional, and the Challenge is a take-home.

**AI in this lab:** you may ask Colab to *explain* an error or a line of code (click **Explain error**, or select a line and ask Gemini to explain it). Do not ask it to write or fix code, and turn off AI code completion. This lab has no AI coding section: you write every line yourself. The one use of AI is drafting your README from the prompt at the end.

---

> ### What you write, and what you run
>
> **You write:**
> - **Part 4, Steps 11-13:** five blanks (`____`). In Step 11, three one-line blanks: the two groups' retention arrays and the difference between their rates. In Steps 12 and 13, one call each: the permutation test and the bootstrap interval, modelled on Part 2's calls.
> - **Write it yourself (required),** after Part 4: `n_needed()`, a function that turns an effect and a standard deviation into the sample size per group for 80% power, then one call to it on the Cookie Cats numbers and a print. About six lines from a blank cell. The check cell below it must print "Passed".
> - The answers to the interpretation questions after each part, in text cells.
>
> **You run and read:** everything else. Setup, Parts 1-3, Step 10, and the Extension's simulation are working code to learn from. Read each cell before you run it, and make sure you can say what each line does. The Extension (optional) has a metric-design table to fill in as text, and asks you to change one line of the simulation, run it again, and answer in one sentence.
>
> **The Challenge** (take-home) has three blanks in a given simulation, and a written reflection. The **README** is text only: the prompt at the end is the one place you use AI, and only for documentation.

## Setup

In [ ]:
# GUIDED — run as-is
# Step 1: install statsmodels, then import libraries
!pip install -q statsmodels

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.power import TTestIndPower
from pathlib import Path

# Every resampling call below uses this seed, so your numbers match the expected output
RANDOM_STATE = 42

# savefig will not create a folder, and a fresh Colab session has no figures/ folder
Path("figures").mkdir(exist_ok=True)

print("Setup complete.")

## Part 1: The Experimental Benchmark — LaLonde NSW (GUIDED — 10 min)

In Lab 8, you tested whether treated and control earnings differed using a t-test.
The LaLonde data come from a **randomized experiment**: random assignment decided who got
the training. That is what lets us read the difference in means as the **causal effect** of
the training. This part shows what goes wrong without it.

We will:
1. Compute the naive ATE (difference in means) from the experimental data
2. Simulate what happens when we **lose** random assignment (observational comparison)
3. Decompose the gap into the true treatment effect + selection bias

In [ ]:
# GUIDED — run as-is
# Step 2: load the NSW experiment and compute the experimental ATE

# Dehejia & Wahba (2002) version of LaLonde's data, served by NBER; .dta is a Stata file
url = "https://users.nber.org/~rdehejia/data/nsw_dw.dta"
df = pd.read_stata(url)

# treat is 1 for a trainee and 0 for a control; re78 is earnings in 1978, the outcome
treated = df[df["treat"] == 1]
control = df[df["treat"] == 0]

ate_experimental = treated["re78"].mean() - control["re78"].mean()

# ,.2f prints commas between thousands and 2 decimals
print(f"Shape: {df.shape}")
print(f"Treatment group: n={len(treated)}, Control group: n={len(control)}")
print("\nExperimental Average Treatment Effect")
print(f"Mean earnings (treated):  ${treated['re78'].mean():,.2f}")
print(f"Mean earnings (control):  ${control['re78'].mean():,.2f}")
print(f"ATE (experimental):       ${ate_experimental:,.2f}")

**Expected output:** 445 people, 185 trainees and 260 controls. Mean 1978 earnings are \$6,349.14
and \$4,554.80, so the experimental ATE is \$1,794.34: the same difference you tested in Lab 8.
Because random assignment decided who was trained, the two groups differ only by chance and by
the training, which is why this difference can be read as the effect of the training.

**Step 3 asks what happens without random assignment.** LaLonde replaced the randomized controls with
a non-experimental comparison group: workers from a general survey, such as the Panel Study of
Income Dynamics (PSID), who never applied to the program. We do not load the PSID here. Step 3
*simulates* such a group instead: 260 made-up earnings drawn from a lognormal distribution, a
right-skewed shape common for earnings, centred on a median of \$15,000. Its long right tail pulls
its mean above \$20,000, far above the experimental controls. The numbers are invented; they show the
mechanism, not LaLonde's actual estimates.

In [ ]:
# GUIDED — run as-is
# Step 3: a simulated observational comparison group, and the selection bias it brings

np.random.seed(42)          # the same made-up earnings on every run

# lognormal draws are e raised to a normal draw, so mean=np.log(15000) puts the median at $15,000
n_comparison = len(control)
comparison_earnings = np.random.lognormal(mean=np.log(15000), sigma=0.8, size=n_comparison)

# The same trainees, compared with the simulated group instead of the randomized controls
ate_observational = treated["re78"].mean() - comparison_earnings.mean()
selection_bias = ate_observational - ate_experimental

# >10,.2f right-aligns each number in 10 characters, so the column lines up
print("Selection Bias Decomposition")
print(f"Experimental ATE (causal):    ${ate_experimental:>10,.2f}")
print(f"Observational ATE (biased):   ${ate_observational:>10,.2f}")
print(f"Selection Bias:               ${selection_bias:>10,.2f}")
print(f"\nFormula: Obs. ATE = True ATE + Selection Bias")
print(f"         {ate_observational:,.0f}  =  {ate_experimental:,.0f}  +  ({selection_bias:,.0f})")

**Expected output:** an observational ATE of −\$14,784.45 and a selection bias of −\$16,578.80. The
last line checks the decomposition with rounded numbers: −14,784 = 1,794 + (−16,579).

In [ ]:
# GUIDED — run as-is
# Step 4: the three earnings distributions, and the two estimates of one effect

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# density=True scales each histogram to area 1, so groups of different sizes compare fairly;
# alpha=0.5 makes the bars see-through, so the overlap shows
axes[0].hist(treated["re78"], bins=30, alpha=0.5, density=True, color="steelblue",
             label=f"Treated (n={len(treated)})")
axes[0].hist(control["re78"], bins=30, alpha=0.5, density=True, color="peru",
             label=f"Exp. Control (n={len(control)})")
axes[0].hist(comparison_earnings, bins=30, alpha=0.5, density=True, color="navy",
             label=f"Obs. Comparison (n={n_comparison})")
axes[0].set_xlabel("1978 Earnings ($)")
axes[0].set_ylabel("Density")
axes[0].set_title("Treatment vs. Control vs. Comparison Group")
axes[0].legend()

axes[1].bar(["Experimental\n(Causal)", "Observational\n(Biased)"],
            [ate_experimental, ate_observational], color=["steelblue", "peru"], width=0.5)
axes[1].axhline(y=0, color="gray", linewidth=0.5)
# An arrow from the label to the top of the observational bar
axes[1].annotate(f"Selection Bias\n= ${selection_bias:,.0f}", xy=(1, ate_observational),
                 xytext=(0.5, ate_experimental / 2), color="red",
                 arrowprops={"arrowstyle": "->", "color": "red"})
axes[1].set_ylabel("Estimated ATE ($)")
axes[1].set_title("Why Random Assignment Matters")

plt.tight_layout()
plt.savefig("figures/ch09_selection_bias.png", dpi=150, bbox_inches="tight")
plt.show()

**What to look for:** on the left, where the simulated comparison group sits against the two
experimental groups. On the right, two estimates of the effect of the same program on the same
trainees.

### Interpretation Questions

1. The experimental ATE is positive, but the observational ATE is negative. What caused the sign flip? In plain language, what does "selection bias" mean here?
2. If you were a policymaker who only had the observational estimate, what conclusion would you draw about the job training program? Why would that conclusion be wrong?
3. LaLonde (1986) showed that econometric methods (regression, matching) could not reliably recover the experimental benchmark from observational data. Why is this finding so important for causal inference?

*Your answers here:*

1. 
2. 
3. 

## Part 2: Permutation Test — Is the ATE Real or Noise? (GUIDED — 15 min)

The t-test from Lab 8 assumes normally distributed data or large samples (CLT).
The **permutation test** makes no distributional assumptions. It asks:
*"If treatment had no effect, how often would we see an ATE this large just by chance?"*

The logic: if treatment labels are meaningless (null is true), then shuffling them
should produce ATEs that look similar to what we observed. If our observed ATE
is extreme relative to the shuffled distribution, the null is implausible.

`stats.permutation_test` does the shuffling. You give it the two groups and a function that
computes the statistic, here the difference in means. scipy hands that function many shuffles at
once, one per row, and passes `axis` to say which way to average; the function only has to pass
`axis` on to `np.mean`.

In [ ]:
# GUIDED — run as-is
# Step 5: permutation test on the LaLonde experiment

def ate_statistic(x, y, axis):
    """Difference in means, x minus y, along axis."""
    return np.mean(x, axis=axis) - np.mean(y, axis=axis)


# .values turns a column into a plain NumPy array, which is what scipy expects
treated_earnings = treated["re78"].values
control_earnings = control["re78"].values

# "independent" shuffles the treat/control labels across all 445 people, 9,999 times
perm_result = stats.permutation_test(
    (treated_earnings, control_earnings),
    statistic=ate_statistic,
    permutation_type="independent",
    n_resamples=9999,
    random_state=RANDOM_STATE,
    alternative="two-sided"
)

print("Permutation Test")
print(f"Observed ATE:     ${ate_experimental:,.2f}")
print(f"Permutation p:    {perm_result.pvalue:.4f}")
print(f"N resamples:      {len(perm_result.null_distribution):,}")
print(f"\nDecision at alpha=0.05: {'Reject H0' if perm_result.pvalue < 0.05 else 'Fail to reject H0'}")

**Expected output:** p = 0.0046 from 9,999 shuffles: reject H0 at alpha = 0.05.

In [ ]:
# GUIDED — run as-is
# Step 6: the null distribution, with the observed ATE marked

fig, ax = plt.subplots(figsize=(10, 6))

# One ATE for each of the 9,999 shuffles: what chance alone produces
ax.hist(perm_result.null_distribution, bins=50, color="steelblue", edgecolor="white",
        label="Null distribution\n(shuffled labels)")
# A two-sided test counts both tails, so the mirror image of the observed ATE is marked too
ax.axvline(ate_experimental, color="peru", linewidth=2.5, linestyle="--",
           label=f"Observed ATE = ${ate_experimental:,.0f}")
ax.axvline(-ate_experimental, color="peru", linewidth=2.5, linestyle=":",
           label=f"Mirror = -${ate_experimental:,.0f}")
ax.set_xlabel("ATE Under Null ($)")
ax.set_ylabel("Frequency")
ax.set_title("Permutation Test: Null Distribution vs. Observed ATE")
ax.legend(loc="upper left")

plt.tight_layout()
plt.savefig("figures/ch09_permutation_null.png", dpi=150, bbox_inches="tight")
plt.show()

# np.abs drops the sign, so this counts both tails; True counts as 1 in a sum, and
# .1% multiplies by 100 and adds a percent sign
far = np.abs(perm_result.null_distribution) >= abs(ate_experimental)
print(f"Shuffled ATEs at least as far from zero as the observed one: {far.sum()} of {len(far):,} ({far.mean():.1%})")

**Expected output:** the shuffled ATEs pile up around zero, and 44 of the 9,999 (0.4%) land beyond
one of the two lines. That share is almost the p-value from Step 5. scipy's 0.0046 differs a little
because it counts the observed split as one more possible shuffle, and doubles the smaller of the
two tails.

In [ ]:
# GUIDED — run as-is
# Step 7: bootstrap 95% confidence interval for the ATE

# The permutation test asks whether the effect could be chance. The bootstrap asks how big it
# plausibly is: it resamples each group with replacement and recomputes the ATE each time.
# ate_statistic from Step 5 is the statistic again.
boot_result = stats.bootstrap(
    (treated_earnings, control_earnings),
    statistic=ate_statistic,
    n_resamples=9999,
    confidence_level=0.95,
    method="BCa",          # bias-corrected and accelerated: adjusts the interval for skewed data
    random_state=RANDOM_STATE
)

ci_low = boot_result.confidence_interval.low
ci_high = boot_result.confidence_interval.high

print("Bootstrap 95% BCa Confidence Interval")
print(f"Point estimate (ATE):  ${ate_experimental:,.2f}")
print(f"95% CI:                [${ci_low:,.2f}, ${ci_high:,.2f}]")

**Expected output:** a 95% BCa interval from \$575.18 to \$3,262.37 around the point estimate of
\$1,794.34.

### Interpretation Questions

1. The permutation test makes no assumption about the shape of the earnings distribution. Why is this an advantage over the t-test for skewed data like earnings?
2. Compare the permutation p-value to the t-test p-value from Lab 8. Are they similar? Why or why not?
3. The bootstrap CI gives a range for the ATE. If you were advising a policymaker, would you report the point estimate alone or the full CI? Why?

*Your answers here:*

1. 
2. 
3. 

## Part 3: Power Analysis — Could We Have Detected the Effect? (GUIDED — 15 min)

Power is the chance that a test detects an effect of a given size, if the effect is real.
Below we compute it for the LaLonde study, find the sample size needed for three dollar
effects, and plot power against sample size.

`solve_power` takes an effect in standard deviations (Cohen's d, from Lab 8), so each dollar
effect below is divided by the pooled standard deviation first.

In [ ]:
# GUIDED — run as-is
# Step 8: Cohen's d, and the power our study achieved

power_analysis = TTestIndPower()

# Cohen's d, as in Lab 8. pandas' .std() divides by n - 1, the sample standard deviation
n_t, n_c = len(treated), len(control)
s_t = treated["re78"].std()
s_c = control["re78"].std()
s_pooled = np.sqrt(((n_t - 1) * s_t**2 + (n_c - 1) * s_c**2) / (n_t + n_c - 2))
d_lalonde = ate_experimental / s_pooled

# The chance our test detects an effect of size d, with our group sizes; ratio is n_c / n_t
achieved_power = power_analysis.solve_power(
    effect_size=abs(d_lalonde),
    nobs1=n_t,
    alpha=0.05,
    ratio=n_c / n_t,
    alternative="two-sided"
)

print("Power of the LaLonde Study")
print(f"Cohen's d:           {d_lalonde:.3f}")
print(f"Treatment n:         {n_t}")
print(f"Control n:           {n_c}")
print(f"Achieved power:      {achieved_power:.3f} ({achieved_power*100:.1f}%)")
print(f"Target:              0.800 (80%)")

**Expected output:** d = 0.273 and an achieved power of 0.808, the same as Lab 8, since it is the
same data.

In [ ]:
# GUIDED — run as-is
# Step 8, continued: the sample size needed for three dollar effects

# Leave nobs1 out and give power instead: solve_power then solves for the n per group.
# n_by_effect is a dictionary: n_by_effect[500] will hold the n for a $500 effect
print(f"Required Sample Size per Group (80% power, alpha=0.05)")
n_by_effect = {}
for dollar_effect in [500, 1000, 2000]:
    d_target = dollar_effect / s_pooled
    n_required = power_analysis.solve_power(
        effect_size=d_target,
        alpha=0.05,
        power=0.80,
        ratio=1.0,
        alternative="two-sided"
    )
    n_by_effect[dollar_effect] = n_required
    # >5, pads to 5 characters with a thousands comma; >6,.0f does the same with no decimals
    print(f"  To detect ${dollar_effect:>5,} effect (d={d_target:.3f}): n = {n_required:>6,.0f} per group ({2*n_required:,.0f} total)")

print(f"\nn for a $500 effect / n for a $2,000 effect: {n_by_effect[500] / n_by_effect[2000]:.0f}")

**Expected output:** a \$500 effect needs 2,719 people per group, \$1,000 needs 681, and \$2,000
needs 171. The last line divides the first of those by the last: 16. Compare that with the ratio
of the two effects. In practice you round each n up, since you cannot recruit part of a person.

In [ ]:
# GUIDED — run as-is
# Step 9: power curves, the power for each n per group, at three dollar effects

fig, ax = plt.subplots(figsize=(10, 6))
sample_sizes = np.arange(20, 1000)      # 20, 21, ..., 999 per group

for dollar_eff, color in [(500, "peru"), (1000, "navy"), (2000, "steelblue")]:
    d = dollar_eff / s_pooled
    # One power for every n in sample_sizes, as in Lab 8's power curves
    powers = [power_analysis.solve_power(
        effect_size=d, nobs1=n, alpha=0.05,
        ratio=1.0, alternative="two-sided"
    ) for n in sample_sizes]
    ax.plot(sample_sizes, powers, label=f"${dollar_eff:,} effect (d={d:.2f})", linewidth=2, color=color)

ax.axhline(y=0.80, color="gray", linestyle="--", label="80% power target")
ax.axvline(x=n_t, color="red", linestyle=":", label=f"Our n={n_t}")
ax.set_xlabel("Sample Size per Group")
ax.set_ylabel("Statistical Power")
ax.set_title("Power Curves — Planning an A/B Test Before You Run It")
ax.legend(loc="right")
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("figures/ch09_power_curves.png", dpi=150, bbox_inches="tight")
plt.show()

**Reading the chart.** Each curve is one dollar effect, turned into d with the pooled SD from
Step 8. Where a curve crosses the dashed 80% line is the n per group that Step 8 printed for that
effect: 171 for \$2,000 and 681 for \$1,000. The \$500 curve needs 2,719, off the right edge of the
chart. The red line marks our 185 trainees.

### Interpretation Questions

1. Was the LaLonde study adequately powered (>80%) to detect the effect it found? What would happen if the true effect were half as large?
2. Step 8 printed the n per group needed to detect a \$500 effect on earnings. A tech company wants to detect a \$500 increase in revenue per user. Can it use that n for its own test? What would it need to measure first, and why?
3. Why should power analysis be done BEFORE running an experiment, not after? What goes wrong with "post-hoc" power analysis?

*Your answers here:*

1. 
2. 
3. 

## Part 4: Cookie Cats A/B Test (YOUR TASK — 25 min)

Now you run the same three steps (difference in means, permutation test, bootstrap) on a
real A/B test from a mobile game. **Cookie Cats** is a mobile
puzzle game where players hit "gates" that force them to wait (or pay). The company
ran an A/B test: should the first gate appear at level 30 or level 40?

- **Treatment groups:** `gate_30` (control) vs. `gate_40` (variant)
- **Outcome:** `retention_1` — did the player return the next day? (binary: 0/1)
- **Question:** Does moving the gate from level 30 to 40 affect Day-1 retention?

Step 10 loads the data. Then your tasks:
1. Compute the observed difference in retention rates (Step 11)
2. Run a permutation test to assess statistical significance (Step 12)
3. Compute a bootstrap 95% CI for the difference (Step 13)

Steps 12 and 13 can each take up to a minute to run: every shuffle and every resample covers all
90,189 players.

In [ ]:
# GUIDED — run as-is
# Step 10: load the Cookie Cats data

cookie_url = "https://raw.githubusercontent.com/Saikushal185/ab-testing-analytics/main/data/cookie_cats.csv"
cc = pd.read_csv(cookie_url)

print(f"Shape: {cc.shape}")
print(f"\nGroup sizes:")
print(cc["version"].value_counts())
print(f"\nFirst 5 rows:")
cc.head()

**Expected output:** 90,189 players: 44,700 in `gate_30` and 45,489 in `gate_40`.

In [ ]:
# YOUR TASK — fill in the three blanks (____)
# Step 11: split the players by version, and compute the difference in Day-1 retention

# Blanks 1 and 2: each group's retention_1 values, as a NumPy array.
# Step 2 split LaLonde on its treat column, and Step 5 took .values; split on version here
gate_30_retention = ____
gate_40_retention = ____

# Blank 3: gate_30's retention rate minus gate_40's, so a positive number means gate_30 kept more.
# retention_1 is 1 for a player who came back and 0 otherwise, so its mean is the share who came back
observed_diff = ____

print("Cookie Cats Day-1 Retention")
print(f"gate_30 retention rate: {np.mean(gate_30_retention):.4f} ({np.mean(gate_30_retention)*100:.2f}%)")
print(f"gate_40 retention rate: {np.mean(gate_40_retention):.4f} ({np.mean(gate_40_retention)*100:.2f}%)")
print(f"Difference (gate_30 - gate_40): {observed_diff:.4f} ({observed_diff*100:.2f} pp)")
print(f"\nn_gate_30 = {len(gate_30_retention)}, n_gate_40 = {len(gate_40_retention)}")

**Expected output:** a retention rate of 0.4482 for `gate_30` and 0.4423 for `gate_40`: a
difference of 0.0059, or 0.59 percentage points (pp). *NameError: name '____' is not defined*
means a blank is still empty.

In [ ]:
# YOUR TASK — fill in the blank (____)
# Step 12: permutation test on Cookie Cats retention, and its null distribution

# Blank: the permutation test from Step 5, on the two retention arrays, gate_30 first.
# Add batch=100 to the call. Without it, scipy builds one 9,999 x 90,189 array and a free
# Colab session crashes for lack of memory; batch=100 does a hundred shuffles at a time.
cc_perm_result = ____

print("Cookie Cats Permutation Test")
print(f"Observed difference: {observed_diff:.4f}")
print(f"Permutation p-value: {cc_perm_result.pvalue:.4f}")

# The null distribution, with the observed difference marked, as in Step 6
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(cc_perm_result.null_distribution, bins=50, color="steelblue", edgecolor="white",
        label="Null distribution")
ax.axvline(observed_diff, color="peru", linewidth=2.5, linestyle="--",
           label=f"Observed diff = {observed_diff:.4f}")
ax.set_xlabel("Retention Difference Under Null")
ax.set_ylabel("Frequency")
ax.set_title("Cookie Cats: Permutation Test Null Distribution")
ax.legend()
plt.tight_layout()
plt.savefig("figures/ch09_cookie_cats_permutation.png", dpi=150, bbox_inches="tight")
plt.show()

**Expected output:** p = 0.0752.

In [ ]:
# YOUR TASK — fill in the blank (____)
# Step 13: bootstrap 95% BCa confidence interval for the retention difference

# Blank: the bootstrap from Step 7, on the two retention arrays, gate_30 first. Add batch=100
# here too. BCa also runs a jackknife over all 90,189 players; unbatched, this cell needs about
# 9.5 GB, and a free Colab session has under 13 GB with everything else already loaded.
cc_boot_result = ____

cc_ci_low = cc_boot_result.confidence_interval.low
cc_ci_high = cc_boot_result.confidence_interval.high

print("Cookie Cats Bootstrap 95% BCa CI")
print(f"Point estimate:  {observed_diff:.4f} ({observed_diff*100:.2f} pp)")
print(f"95% CI:          [{cc_ci_low:.4f}, {cc_ci_high:.4f}]")
print(f"                 [{cc_ci_low*100:.2f} pp, {cc_ci_high*100:.2f} pp]")

**Expected output:** an interval from −0.0007 to 0.0123, or −0.07 pp to 1.23 pp.

### Interpretation Questions

1. Based on your permutation test and bootstrap CI, should the game company move the gate from level 30 to level 40? What is your recommendation and why?
2. The Cookie Cats dataset has about 45,000 players per group. The LaLonde study had about 200 per group. How does this difference in sample size affect the precision of your estimates?
3. The retention difference is small in absolute terms (a fraction of a percentage point). When might a tiny effect still justify a product change? When might it not?

*Your answers here:*

1. 
2. 
3. 

---

## Write it yourself (required — 20 min)

This part is required, and it is short. The check cell below it must print "Passed" before you
submit.

**The task.** Suppose the company believes the Cookie Cats difference is real and wants to run the
test again, large enough to detect it. How many players per group would that take? Step 8 answered
the same question for LaLonde's dollar effects. Turn Step 8's calculation into a function, then
point it at Cookie Cats.

Write a function `n_needed(effect, sd)` that takes an effect and the outcome's standard deviation,
and gives back the number of people per group needed to detect that effect with 80% power, at
alpha = 0.05, two-sided, with two groups of equal size. Then call it with Step 11's
`observed_diff` as the effect and the standard deviation of `retention_1` across all players,
`np.std(cc["retention_1"], ddof=1)`, as the SD. Store the result as `n_cookie`, and print it.

- It takes about six lines: the `def` line, a docstring, a line or two inside, the call, and a print.
- Step 8's loop already does this for one dollar effect at a time. Look at what changes from one
  effect to the next, and what stays fixed.
- `solve_power` measures the effect in standard deviations, not in dollars or in shares.
- Lab 8's `welch_t()` shows the shape of a function: the `def` line, a one-line docstring, the
  body, and `return`.

Then run the check cell. It calls your function on Step 8's three dollar effects and compares the
answers with Step 8's, and once more with the SD doubled. Then it checks `n_cookie`. `assert` stops with its message when its
condition is False; the message tells you what to fix.

In [ ]:
# WRITE IT YOURSELF (required): n_needed(), then n_cookie for Cookie Cats

In [ ]:
# CHECK — for "Write it yourself" above

def as_number(x):
    """When solve_power finds no answer it can give back a one-element array; this takes the number out."""
    return float(np.ravel(x)[0])


# globals() holds every name defined so far in this notebook
if "n_needed" not in globals():
    print("Not written yet: write n_needed() in the cell above and run it, then run this check.")
else:
    result = n_needed(500, s_pooled)
    assert result is not None, "n_needed gave back nothing. Does it end with a return line?"
    yours = as_number(result)
    step_8 = n_by_effect[500]
    share = yours / step_8          # your answer as a share of Step 8's

    # nan means "not a number": solve_power found no answer
    assert not np.isnan(yours), (
        "n_needed(500, s_pooled) gives nan. solve_power wants the effect in standard deviations: "
        "divide the effect by the SD, as Step 8 does.")
    assert yours >= 1, (
        f"{yours:.3f} is a probability, a power. Give power=0.80 and leave nobs1 out, as Step 8's loop does.")
    assert yours >= 100, (
        f"{yours:,.0f} per group is far too few. solve_power wants the effect in standard deviations: "
        "divide the effect by the SD, as Step 8 does.")
    assert yours < 1e6, (
        f"{yours:,.0f} per group is far too many: divide by the SD, not the variance.")
    # Three slips land at a fixed share of Step 8's answer: the total for both groups at twice it,
    # a one-sample test at about half, and a one-sided test (or alpha = 0.10) at about 79%
    assert not 1.99 < share < 2.01, (
        "That is the total for both groups. Give back the n for one group, as solve_power does.")
    assert not 0.49 < share < 0.51, (
        f"{yours:,.0f} is the one-sample answer. Use power_analysis from Step 8, which compares two groups.")
    assert not 0.77 < share < 0.80, (
        f"{yours:,.0f} is the one-sided answer at alpha 0.05, which is also the two-sided answer at "
        "0.10. Step 8 uses alpha = 0.05, two-sided.")
    assert abs(yours - step_8) <= 1, (
        f"n_needed(500, s_pooled) gives {yours:,.1f}, and Step 8 gives {step_8:,.1f} for a $500 "
        "effect. Check alpha, power and ratio against Step 8's loop; the groups are of equal size.")

    # Twice the effect with twice the SD is the same effect in SDs, so the same n as $500
    assert abs(as_number(n_needed(1000, 2 * s_pooled)) - step_8) <= 1, (
        "n_needed(1000, 2*s_pooled) should equal the $500 answer. Does it divide by its sd argument, "
        "or by s_pooled?")
    # Step 8's other two effects, to be sure the function uses its effect argument
    for effect in [1000, 2000]:
        n_e = as_number(n_needed(effect, s_pooled))
        assert abs(n_e - n_by_effect[effect]) <= 1, (
            f"Right for $500, but for ${effect:,} you get {n_e:,.1f} and Step 8 has "
            f"{n_by_effect[effect]:,.1f}. Does n_needed use its effect argument, or a number typed in?")

    assert "n_cookie" in globals(), (
        "Your function works. Now call it on the Cookie Cats numbers and store the result as n_cookie.")
    # The Cookie Cats numbers, computed here from the data, not from your Step 11;
    # groupby gives the mean retention of each version
    by_version = cc.groupby("version")["retention_1"].mean()
    cc_effect = by_version["gate_30"] - by_version["gate_40"]
    cc_sd = np.std(cc["retention_1"], ddof=1)
    target = as_number(n_needed(cc_effect, cc_sd))      # your function has passed every test above
    mine = as_number(n_cookie)
    assert mine > 1000, (
        f"n_cookie is {mine:,.0f}, far too small. Are the effect and the SD in the same units? "
        "observed_diff is a share (about 0.006), not percentage points.")
    assert abs(mine - target) / target < 0.001, (
        f"n_cookie is {mine:,.0f}, and the check gets {target:,.0f}. Use Step 11's observed_diff as "
        "the effect and np.std(cc[\"retention_1\"], ddof=1) as the SD.")
    print("Passed. n_needed matches Step 8 for all three dollar effects, and uses its sd argument.")
    print(f"Players per group to detect a {cc_effect * 100:.2f} pp difference: {mine:,.0f}")

---

## Extension (Optional — 15 min): Metric Design & the Peeking Problem

Two more things to get right in an A/B test:
1. **Metric Design:** Choosing the right outcome metric (and guardrails) before the test
2. **The Peeking Problem:** What happens when you check results too early and too often

### Part A: Metric Design Exercise

**Scenario:** An e-commerce company is testing a new checkout flow. The new flow has
fewer steps (3 instead of 5) but removes the "save for later" option.

**Your task:** Define the following metrics for this A/B test. Be specific about
how each would be measured.

| Metric Type | Definition | Your Metric |
|-------------|-----------|-------------|
| **Primary metric** | The one thing you most want to improve | *Your answer here* |
| **Guardrail metric** | Something that must NOT get worse | *Your answer here* |
| **Counter-metric** | A metric that might improve artificially (Goodhart's Law risk) | *Your answer here* |

**Additional questions:**
1. Why is it dangerous to have only a primary metric without guardrails?
2. If the primary metric improves by 5% but the guardrail metric drops by 2%, what do you recommend?

*Your answers here:*

1. 
2. 

### Part B: The Peeking Problem — Why Early Stopping Inflates False Positives

In practice, data scientists are tempted to "peek" at A/B test results before the
test reaches its planned sample size. Each look is another chance of a false positive. The
simulation below counts how many.

The simulation runs 1,000 **A/A tests**: both groups come from the same 10% conversion rate, so
there is no effect to find, and every "significant" result is a false positive. Each test is
analysed two ways. *Peeking* checks the p-value at 50, 100, 200 and 500 users per group and stops
at the first p < 0.05. *Fixed* checks once, at the planned 500. The p-value comes from a
chi-squared test on the 2 x 2 table of conversions and non-conversions in each group, which is
the standard test that two conversion rates are equal.

In [ ]:
# EXTENSION — run as-is
# Step 14: 1,000 A/A tests, analysed with peeking and without

def chi2_p(a, b):
    """p-value of a chi-squared test that two 0/1 samples share one conversion rate."""
    table = [[a.sum(), len(a) - a.sum()],
             [b.sum(), len(b) - b.sum()]]
    # correction=False gives the plain test; chi2_contingency gives back several things, [1] is the p-value
    return stats.chi2_contingency(table, correction=False)[1]


np.random.seed(42)          # the same simulated tests on every run
n_experiments = 1000
peek_points = [50, 100, 200, 500]   # users per group at each look
baseline_rate = 0.10                # both groups convert at 10%: no true effect

false_positives_peek = 0
false_positives_fixed = 0

# _ stands for the loop counter, which we never use
for _ in range(n_experiments):
    # binomial(1, rate, size) draws 0/1 outcomes: 1 converts, with probability rate
    group_a = np.random.binomial(1, baseline_rate, size=max(peek_points))
    group_b = np.random.binomial(1, baseline_rate, size=max(peek_points))

    # Peeking: test the first n users at each look; break leaves the loop at the first p < 0.05
    for n in peek_points:
        if chi2_p(group_a[:n], group_b[:n]) < 0.05:
            false_positives_peek += 1
            break

    # Fixed: one test, on all the users
    if chi2_p(group_a, group_b) < 0.05:
        false_positives_fixed += 1

fp_rate_peek = false_positives_peek / n_experiments
fp_rate_fixed = false_positives_fixed / n_experiments

print("The Peeking Problem: Simulated False Positive Rates")
print(f"Number of experiments: {n_experiments}")
print(f"Peek checkpoints:      n = {peek_points}")
print(f"\nFixed analysis (check only at n={max(peek_points)}):")
print(f"  False positive rate: {fp_rate_fixed:.3f} ({fp_rate_fixed*100:.1f}%)")
print(f"  Expected:            0.050 (5.0%)")
print(f"\nPeeking analysis (stop at first p<0.05):")
print(f"  False positive rate: {fp_rate_peek:.3f} ({fp_rate_peek*100:.1f}%)")
print(f"  Inflation factor:    {fp_rate_peek/0.05:.1f}x the nominal 5% rate")

**Expected output:** the fixed analysis has a false positive rate of 0.057, close to the 5% that
alpha = 0.05 promises. Peeking at four checkpoints gives 0.156, 3.1 times the nominal rate, on
data with no effect at all. This is why A/B testing platforms use sequential testing methods,
which are built to allow repeated looks.

**Try it.** Add two more checkpoints, 300 and 400, to `peek_points` and run the cell again. Report
the new false positive rate with peeking, and say in one sentence whether the damage grows with
the *number* of peeks or with how *early* the first one is.

*Your answer here:*

---

## Challenge (Take-Home): Design and Simulate a Full A/B Test

You are the data scientist for an e-commerce company. The product team wants to test
a new checkout button color. You believe it will increase conversion from 10% to 12%
(delta = 0.02 on a 10% baseline).

Your tasks:
1. Run a power analysis to determine the required sample size per group
2. Simulate the experiment 100 times with the KNOWN true effect (delta = 0.02), at that sample size
3. Run a permutation test on each simulated experiment
4. Compare the average estimated ATE to the true ATE, and the share of tests that reject H0 to the
   power you designed for
5. Plot the distribution of the 100 estimated ATEs

For two proportions, the effect size is **Cohen's h**, not Cohen's d:

$$h = 2\arcsin\sqrt{p_1} - 2\arcsin\sqrt{p_2}$$

where $p_1$ is the treatment rate and $p_2$ the baseline. In NumPy, $\arcsin$ is `np.arcsin`.
`TTestIndPower` accepts h in place of d; the answer is an approximation, and close.

If you skip the Challenge, leave its cell unrun, so your submitted notebook shows no error.

In [ ]:
# YOUR TASK — take-home Challenge: fill in the three blanks (____)

np.random.seed(42)
baseline_rate = 0.10
true_delta = 0.02            # the true effect: 10% -> 12%
treatment_rate = baseline_rate + true_delta

# Blank 1: Cohen's h for treatment_rate against baseline_rate (the formula is above)
h = ____
# Blank 2: the n per group for 80% power at alpha = 0.05, two-sided; Step 8 shows the call
n_required = ____
n_sim = int(np.ceil(n_required))      # np.ceil rounds up: you cannot recruit part of a person

estimated_ates = []
p_values = []
for _ in range(100):
    control_sim = np.random.binomial(1, baseline_rate, size=n_sim)
    treatment_sim = np.random.binomial(1, treatment_rate, size=n_sim)
    estimated_ates.append(np.mean(treatment_sim) - np.mean(control_sim))
    # Blank 3: Step 5's permutation test, treatment first, with n_resamples=999 so that
    # the 100 tests finish in under a minute
    sim_result = ____
    p_values.append(sim_result.pvalue)

print(f"Cohen's h: {h:.4f}   n per group: {n_sim:,}")
print(f"True ATE: {true_delta:.4f}   Mean of 100 estimated ATEs: {np.mean(estimated_ates):.4f}")
# p < 0.05 is True or False for each test, and the mean of True/False is the share that are True
print(f"Share of the 100 tests that reject H0: {np.mean(np.array(p_values) < 0.05):.2f}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(estimated_ates, bins=20, color="steelblue", edgecolor="white")
ax.axvline(true_delta, color="red", linestyle="--", label=f"True ATE = {true_delta}")
ax.set_xlabel("Estimated ATE")
ax.set_ylabel("Frequency")
ax.set_title("100 Simulated A/B Tests")
ax.legend()
plt.show()

**Reflect.** Is the average estimated ATE close to the true one? Is the share of tests that reject
H0 close to the 80% power you designed for? If it differs, say whether 100 tests is enough to tell.
Then halve the true effect to `true_delta = 0.01`, run the cell again, and say what happens to
`n_sim`. *NameError: name '____' is not defined* means a blank is still empty.

*Your answer here:*

---
## Digital Portfolio: Your README

Copy the prompt below into Claude or ChatGPT. Fill in each [YOUR VALUE] from your own output
first, and do not quote a number this notebook did not print. **Do NOT ask the AI to write
Python code — only documentation.**

```
I need help writing a project README for my data science lab.
**Important Rule:** Do NOT generate any Python code for me.

**What I did in this lab:**
* Loaded the LaLonde NSW job training experiment (445 people:
  185 trainees and 260 controls) and computed the experimental ATE
* Simulated an observational comparison group to show selection bias,
  and split the gap: Observational ATE = True ATE + Selection Bias
* Ran a permutation test (9,999 shuffles) and a bootstrap BCa 95%
  confidence interval for the ATE
* Ran a power analysis: the power of this study, and the sample size
  needed for three dollar effects
* Applied the same tests to the Cookie Cats mobile game A/B test
  (90,189 players, gate at level 30 or 40, Day-1 retention)
* Wrote a sample-size function, n_needed(), checked it against the
  power analysis, and used it to size a rerun of Cookie Cats
* Key findings: trainees earned $[YOUR VALUE] more (permutation
  p = [YOUR VALUE], 95% CI $[YOUR VALUE] to $[YOUR VALUE]); the simulated
  comparison group gave an ATE of $[YOUR VALUE]. Cookie Cats: a
  retention difference of [YOUR VALUE] pp (p = [YOUR VALUE]); a rerun
  would need [YOUR VALUE] players per group

**Please write a README.md entry including:**
1. Project Title: A/B Testing — Designing the Data-Generating Process
2. Objective: one sentence
3. Methodology: Bullet points of technical steps
4. Key Findings: Summary of results
Write it plainly, in the first person. No words like 'robust', 'comprehensive'
or 'leverage'. Use only the numbers I gave you.
```

---

## Submit this lab on GitHub

**Repository name for this lab:** `econ3916-lab09-ab-testing`

**First time?** Read the **GitHub Setup** page in the Start Here module once,
start to finish. It assumes you have never used git and never seen GitHub.

1. On [github.com](https://github.com): **+** (top right) → **New repository**.
   Name it exactly `econ3916-lab09-ab-testing`, tick **Add a README file**, and create it.
2. In Colab: **File → Save a copy in GitHub**. Choose `econ3916-lab09-ab-testing`, branch `main`,
   commit message `Lab 09 submission`. This pushes the notebook, and the
   notebook's rendered output — charts included — is what is graded.
3. Open the repo on github.com, click `README.md`, then the pencil icon, and
   paste in the README you drafted from the prompt above. **Commit
   changes.** Check every number in it against your own output first: this
   goes out under your name.
4. On Canvas: in Colab use **File → Download → Download .ipynb**, upload that
   file to the assignment, and paste your repository URL in the **Comments**
   box beside the upload. Then click **Submit**. Do not use the Website URL
   tab: Canvas keeps one submission, so a URL sent on its own replaces your
   notebook.

No terminal, no token, nothing to install.